# Solução para o problema da mochila usando algoritmos genéticos

In [120]:
import random

In [121]:

NOME =  ["A", "B", "C", "D", "E", "F", "G", "H", "I", "J"]
MASSA =  [2, 3, 4, 5, 7, 1, 6, 4.5, 3.5, 2.5]
VALOR =  [40, 50, 65, 80, 110, 15, 90, 70, 60, 55]

CAPACIDADE = 15

## Iniciando a população inicial

Representação dos cromossomos:

cada cromossomo é um vetor binário v, len(v) = 9.

v[i] = 0 indica a ausência do item de indíce i na mochila, enquanto v[i] = 1 indica a presença de tal item.

In [122]:
populacao = []

while(len(populacao) < 50):
    idv = []
    
    for i in range(9):
        idv.append(random.randint(0, 1)) # gera individuos aleatoriamente

    
    peso_idv = 0
    
    # verifica se o individuo é factivel
    for i in range(9):
        if(idv[i] == 1):
            peso_idv += MASSA[i]
    
    # se é factível, adiciona à população
    if(peso_idv <= CAPACIDADE):
        populacao.append(idv)


## Funções importantes para a execução do algoritmo:

In [123]:
def verifica_factivel(individuo: list) -> int: 

    factivel = 0
    peso_idv = 0

    for i in range(9):
        if(individuo[i] == 1):
            peso_idv += MASSA[i]
        
    if(peso_idv <= CAPACIDADE):

        factivel = 1

    return factivel # 0 se não é factível, 1 se é factível

In [124]:
def calcula_fo(idv: list) -> int: # retorna o valor da funcao objetivo para cada individuo
    
    peso = 0
    valor = 0
    
    for i in range(9):
        if(idv[i] == 1):
            valor += VALOR[i]

    return valor * verifica_factivel(idv)    

In [125]:
def selecao(populacao: list, numero_cruzamento: int) -> list: # retorna os indices dos selecionados para o cruzamento no vetor de populacao
    
    tam = len(populacao)
    idx = [i for i in range(tam)] # vamos retornar os índices dos individuos selecionados


    # calcula a probabilidade de cada item ser selecionado baseado na sua funcao objetivo
    probabilidade = [0] * tam
    selecionados =  [0] * numero_cruzamento

    soma_fos = 0
    
    for i in range(tam):
        soma_fos += calcula_fo(populacao[i])
    
    for i in range(tam):
        probabilidade[i] = (calcula_fo(populacao[i])/soma_fos) * 100 
    
    
    # faz a selecao baseada na probabilidade calculada
    for i in range(numero_cruzamento):

        par = []

        par.append(random.choices(idx, weights = probabilidade)[0])
        par.append(random.choices(idx, weights = probabilidade)[0])


        selecionados[i] = par
    
    
    return selecionados

In [126]:
def mutacao(filhos: list) -> list:

    val = [0, 1]
    chances = [95, 5]

    ocorre = random.choices(val, weights = chances)[0] # 0 se não ocorre mutação, 1 se ocorre

    if (ocorre == 0): return filhos # não houve mutação

    while(True):

        f = random.randint(0, 1) # escolhe aleatoriamente o filho pra mutacao

        gen = random.randint(0, 8) # escolhe o gen que vai ser mudado

        mutante = filhos[f]
        mutante[gen] = (int) (not mutante[gen]) # filho f com o bit do gene especificado invertido

        if(verifica_factivel(mutante) == 0): continue

        filhos[f] = mutante

        return filhos

In [127]:
def gera_nova_populacao(populacao: list, selecionados: list) -> list: # retorna a população com os indivíduos gerados a partir do cruzamento

    for par in selecionados:

        filhos = []

        while(len(filhos) < 2):

            num_genes = random.randint(1, 2) # crossover em 1 ou 2 genes
            

            for i in range(num_genes):

                ######  crossover ######

                # escolhe aleatoriamente os indices dos genes a serem trocado
                gen1 = random.randint(0,8) 
                gen2 = random.randint(0,8)

                print(f"genes que serão trocados: {gen1} e {gen2}")

                #inicia os filhos como copias dos pais
                filho1 = populacao[par[0]]
                filho2 = populacao[par[1]]

                # faz a troca dos genes
                temp = filho1[gen1]
                filho1[gen1] = filho2[gen2]
                filho2[gen2] = temp
 
                ######    fim     ######

                # verifica se filhos são factíveis
                if(verifica_factivel(filho1) == 1): filhos.append(filho1)
                if(verifica_factivel(filho2) == 1): filhos.append(filho2)
        

        filhos = mutacao(filhos) # 5% de chance de ocorrer mutação

        populacao.append(filhos[0])
        populacao.append(filhos[1])

    #retira pais da populacao
    for par in selecionados:
        del populacao[par[0]]
        del populacao[par[1]]        

    return populacao

## Resolução do problema: